In [1]:
import sqlite3
import pandas as pd

# 1. Connect to (or create) a local SQLite database file
conn = sqlite3.connect("../grefton_analytics.db")

# 2. Load our processed data into SQL tables
df_projects = pd.read_csv("../data/processed/clean_projects_financials.csv")
df_clients = pd.read_csv("../data/processed/clean_clients.csv")
df_leads = pd.read_csv("../data/processed/clean_lead_sources.csv")
df_maintenance = pd.read_csv("../data/processed/clean_maintenance.csv")

df_projects.to_sql("fact_projects", conn, if_exists="replace", index=False)
df_clients.to_sql("dim_clients", conn, if_exists="replace", index=False)
df_leads.to_sql("dim_lead_sources", conn, if_exists="replace", index=False)
df_maintenance.to_sql("fact_maintenance", conn, if_exists="replace", index=False)

print("Database grefton_analytics.db initialized with 4 relational tables!\n")


# Helper function to run and format SQL queries neatly
def run_query(query_str, title="Query Result"):
    print(f"=== {title.upper()} ===")
    result = pd.read_sql_query(query_str, conn)
    print(result.to_string(index=False))
    print("\n" + "=" * 60 + "\n")
    return result

Database grefton_analytics.db initialized with 4 relational tables!



In [2]:
# -------------------------------------------------------------------------
# QUERY 1: Service Line Margin Performance & Overrun Breakdown
# Business Goal: Uncover which project types have the highest cost leakage.
# Concepts: GROUP BY, Aggregations, CASE WHEN, ROUND
# -------------------------------------------------------------------------
q1 = """
SELECT 
    project_type,
    COUNT(project_id) AS total_projects,
    ROUND(SUM(contract_value_aed) / 1000000.0, 2) AS total_revenue_millions_aed,
    ROUND(AVG(gross_margin_pct), 2) AS avg_gross_margin_pct,
    ROUND(AVG(cost_overrun_pct), 2) AS avg_cost_overrun_pct,
    SUM(CASE WHEN gross_profit_aed < 0 THEN 1 ELSE 0 END) AS loss_making_projects
FROM fact_projects
WHERE status = 'Completed'
GROUP BY project_type
ORDER BY avg_gross_margin_pct DESC;
"""
run_query(q1, "Service Line Financial Performance")

# -------------------------------------------------------------------------
# QUERY 2: Top 5 Margin-Eroding Projects (Root Cause Diagnosis)
# Business Goal: Identify specific contracts where severe overruns occurred.
# Concepts: Multi-table JOIN, Filtering, ORDER BY
# -------------------------------------------------------------------------
q2 = """
SELECT 
    p.project_id,
    c.client_name,
    c.primary_location,
    p.project_type,
    p.contract_value_aed,
    p.actual_total_cost_aed,
    p.gross_profit_aed,
    p.gross_margin_pct,
    p.subcontractor_pct_of_cost,
    p.schedule_delay_days
FROM fact_projects p
JOIN dim_clients c ON p.client_id = c.client_id
WHERE p.status = 'Completed' AND p.gross_profit_aed < 0
ORDER BY p.gross_profit_aed ASC
LIMIT 5;
"""
run_query(q2, "Top 5 Loss-Making Projects")

# -------------------------------------------------------------------------
# QUERY 3: Client Lifetime Value & AMC Conversion (Window Function)
# Business Goal: Rank our most valuable clients across both installation and maintenance.
# Concepts: CTE (Common Table Expression), DENSE_RANK() Window Function, LEFT JOIN
# -------------------------------------------------------------------------
q3 = """
WITH ClientFinancials AS (
    SELECT 
        c.client_id,
        c.client_name,
        c.client_type,
        COALESCE(SUM(p.contract_value_aed), 0) AS total_project_spend,
        COALESCE(SUM(m.annual_fee_aed), 0) AS total_amc_spend,
        COALESCE(SUM(p.contract_value_aed), 0) + COALESCE(SUM(m.annual_fee_aed), 0) AS total_lifetime_value_aed
    FROM dim_clients c
    LEFT JOIN fact_projects p ON c.client_id = p.client_id
    LEFT JOIN fact_maintenance m ON c.client_id = m.client_id
    GROUP BY c.client_id, c.client_name, c.client_type
)
SELECT 
    client_name,
    client_type,
    total_project_spend,
    total_amc_spend,
    total_lifetime_value_aed,
    DENSE_RANK() OVER (ORDER BY total_lifetime_value_aed DESC) as ltv_rank
FROM ClientFinancials
ORDER BY ltv_rank ASC
LIMIT 10;
"""
run_query(q3, "Top 10 Clients by Lifetime Value (LTV)")

=== SERVICE LINE FINANCIAL PERFORMANCE ===
          project_type  total_projects  total_revenue_millions_aed  avg_gross_margin_pct  avg_cost_overrun_pct  loss_making_projects
      Pool & Hardscape              88                       32.42                 29.86                  2.58                     0
Irrigation & Softscape              66                        9.65                 28.62                  4.80                     0
  Full Villa Landscape              74                       34.06                 28.58                  4.50                     0
      Commercial Plaza              57                       44.69                 27.56                  6.52                     1


=== TOP 5 LOSS-MAKING PROJECTS ===
  project_id             client_name   primary_location     project_type  contract_value_aed  actual_total_cost_aed  gross_profit_aed  gross_margin_pct  subcontractor_pct_of_cost  schedule_delay_days
PRJ-2024-294 Al-Ghurair Holdings 032 Dubai Hills Estate

,client_name,client_type,total_project_spend,total_amc_spend,total_lifetime_value_aed,ltv_rank
0,Private Client 004,Luxury Residential,16557500,1894500,18452000,1
1,Private Client 112,Luxury Residential,6706000,970500,7676500,2
2,Resort & Spa Group 022,Hospitality,6686000,720000,7406000,3
3,Al-Diyar Holdings 009,Commercial Developer,6004000,649500,6653500,4
4,Resort & Spa Group 105,Hospitality,5332500,631500,5964000,5
5,Resort & Spa Group 038,Hospitality,5178000,269000,5447000,6
6,Resort & Spa Group 059,Hospitality,4804500,578750,5383250,7
7,Resort & Spa Group 013,Hospitality,4542000,446250,4988250,8
8,Private Client 100,Luxury Residential,4294000,683000,4977000,9
9,Private Client 021,Luxury Residential,4273000,488000,4761000,10
